In [11]:
from pathlib import Path
import arff
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import numpy as np


# 1. Data Loading

In [3]:
ROOT = Path.cwd()

if ROOT.name == 'notebooks':
    DATA_PATH = ROOT.parent / 'credit.arff'
else:
    DATA_PATH = 'credit.arff'
    
with open(DATA_PATH, 'r') as f:
    arff_data = arff.load(f)
    
data = arff_data['data']
column_names = [attr[0] for attr in arff_data['attributes']]
    
df = pd.DataFrame(data, columns=column_names)
df.head()

,Loan_ID,Loan_Amount_Requested,Length_Employed,Home_Owner,Annual_Income,Income_Verified,Purpose_Of_Loan,Debt_To_Income,Inquiries_Last_6Mo,Months_Since_Deliquency,Number_Open_Accounts,Total_Accounts,Gender,Interest_Rate
0,10139122,"35,000",3 years,NaN,160000.0,VERIFIED - income,credit_card,14.86,1,NaN,6,26,Male,None
1,10025461,"15,000",10+ years,Rent,41000.0,not verified,debt_consolidation,16.51,0,21.0,13,36,Female,None
2,10154747,"11,000",5 years,Mortgage,59000.0,not verified,debt_consolidation,21.75,0,NaN,11,20,Male,None
3,10032437,"12,000",NaN,Mortgage,72000.0,VERIFIED - income,debt_consolidation,15.73,1,NaN,7,20,Female,None
4,10060564,"20,000",< 1 year,Other,79404.0,VERIFIED - income,debt_consolidation,15.32,3,58.0,18,33,Female,None


In [4]:
df['Loan_Amount_Requested'] = df['Loan_Amount_Requested'].str.replace(',', '').astype(float)

In [6]:
X_clf = df.drop(['Purpose_Of_Loan', 'Loan_ID'], axis=1).copy()
y_clf = df['Purpose_Of_Loan'].copy()
y_clf = pd.Series(np.where(y_clf == 'debt_consolidation', 1, 0))

In [12]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

numeric_features = X_clf.select_dtypes(include='number').columns.tolist()
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = X_clf.select_dtypes(exclude='number').columns.tolist()
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

clf_preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

clf_model = Pipeline([
    ('preprocessor', clf_preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

clf_model.fit(X_train_clf, y_train_clf)

proba = clf_model.predict_proba(X_test_clf)[:, 1]
thresholds = [0.3, 0.5, 0.7, 0.9, 0.25]
rows = []
for threshold in thresholds:
    pred = (proba >= threshold).astype(int)
    rows.append({
        'threshold': threshold,
        'accuracy': accuracy_score(y_test_clf, pred),
        'precision': precision_score(y_test_clf, pred),
        'recall': recall_score(y_test_clf, pred),
        'roc_auc': roc_auc_score(y_test_clf, proba)
    })

pd.DataFrame(rows)

/Users/rafalzalecki/studia/coding/sad_lab/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,threshold,accuracy,precision,recall,roc_auc
0,0.30,0.594401,0.594430,0.999744,0.591296
1,0.50,0.610072,0.616022,0.913233,0.591296
2,0.70,0.444698,0.703645,0.113642,0.591296
3,0.90,0.405599,0.000000,0.000000,0.591296
4,0.25,0.594401,0.594430,0.999744,0.591296


If we want to limit false negatives, we should be interested in maximzing recall metric, meaning that 0.3 or 0.25 would be the best option in our case